# Small DiT sanity check

This notebook selects Small DiT through the existing config and GDFM factory.

It calls existing loss and sampling APIs without adding training machinery.

In [ ]:
# Load the existing configuration and public model API.
from pathlib import Path
import sys

repo_root = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "mem_param_src").is_dir()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch

from mem_param_src.config import get_base_config
from mem_param_src.model import GDFM

In [ ]:
# Select the canonical Small DiT image configuration.
cfg = get_base_config()
cfg.network.arch = "dit"
cfg.network.depth = 12
cfg.network.hidden_size = 384
cfg.network.num_heads = 6
cfg.network.mlp_ratio = 4.0
cfg.network.patch_size = 2
cfg.network.learn_sigma = False
cfg.data.image_size = 32
cfg.data.image_channels = 1

In [ ]:
# Construct GDFM and inspect the backbone contract.
model = GDFM(cfg).eval()
z = torch.randn(2, 1, 32, 32)
t = torch.tensor([0.25, 0.75])
with torch.no_grad():
    prediction = model.predict(z, t)
parameter_count = sum(parameter.numel() for parameter in model.net.parameters())
print({"backbone": type(model.net).__name__, "parameters": parameter_count})
print({"input": tuple(z.shape), "times": tuple(t.shape), "output": tuple(prediction.shape)})

In [ ]:
# Call the existing loss and sampling APIs.
loss = model.compute_loss(z, eps=torch.randn_like(z), t=t)
sample = model.sample(
    z[:1],
    n_steps=2,
    method="euler",
    start_time=cfg.sampling.start_time,
    end_time=cfg.sampling.end_time,
    use_safe_div=cfg.sampling.use_safe_div,
    safe_div_denominator_floor=cfg.sampling.safe_div_denominator_floor,
    eps_to_v=cfg.sampling.eps_to_v,
    heun_final_euler=cfg.sampling.heun_final_euler,
    output="x_t_max",
)
print({"loss": float(loss), "sample_shape": tuple(sample.shape)})